In [5]:
# Cell 1 — Load and inspect the flights dataset
import pandas as pd
import os

# Load the data
df_flights = pd.read_csv("../data/raw/flights.csv")

# First look
print("Shape:", df_flights.shape)
print("\nColumns:", df_flights.columns.tolist())
print("\nFirst 3 rows:")

C:\Users\HP\AppData\Local\Temp\ipykernel_23084\119107777.py:6: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  df_flights = pd.read_csv("../data/raw/flights.csv")


Shape: (5819079, 31)

Columns: ['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

First 3 rows:


In [7]:
# Cell 2 — Load cleanly and filter Delta flights only
df_flights = pd.read_csv(
    "../data/raw/flights.csv",
    low_memory=False        # fixes the DtypeWarning
)

# Filter Delta Airlines only (code: DL)
df_delta = df_flights[df_flights["AIRLINE"] == "DL"].copy()

print(f"Total flights in dataset : {len(df_flights):,}")
print(f"Delta flights only       : {len(df_delta):,}")
print(f"Delta % of total         : {len(df_delta)/len(df_flights)*100:.1f}%")

Total flights in dataset : 5,819,079
Delta flights only       : 875,881
Delta % of total         : 15.1%


In [9]:
# Cell 3 — Understand what we are working with
print("=== Dataset Overview ===")
print(f"Rows    : {len(df_delta):,}")
print(f"Columns : {len(df_delta.columns)}")

print("\n=== Missing Values (top 10) ===")
print(df_delta.isnull().sum().sort_values(ascending=False).head(10))

print("\n=== Delay Statistics (minutes) ===")
print(df_delta["DEPARTURE_DELAY"].describe().round(1))

print("\n=== Cancellation Count ===")
print(df_delta["CANCELLED"].value_counts())

=== Dataset Overview ===
Rows    : 875,881
Columns : 31

=== Missing Values (top 10) ===
CANCELLATION_REASON    872057
LATE_AIRCRAFT_DELAY    757858
WEATHER_DELAY          757858
AIRLINE_DELAY          757858
AIR_SYSTEM_DELAY       757858
SECURITY_DELAY         757858
ELAPSED_TIME             5606
AIR_TIME                 5606
ARRIVAL_DELAY            5606
WHEELS_ON                3935
dtype: int64

=== Delay Statistics (minutes) ===
count    872177.0
mean          7.4
std          36.3
min         -61.0
25%          -4.0
50%          -1.0
75%           4.0
max        1289.0
Name: DEPARTURE_DELAY, dtype: float64

=== Cancellation Count ===
CANCELLED
0    872057
1      3824
Name: count, dtype: int64


In [ ]:
# Cancellations: 3,824   ← Delta cancels very few flights
Mean delay:    7.4 min ← Delta is actually quite punctual
Max delay:     1,289 min ← but extreme events still happen #

In [11]:
# Cell 4 — Filter disrupted flights
# Applying Detection Agent thresholds to real data

df_disrupted = df_delta[
    (df_delta["DEPARTURE_DELAY"] >= 30) |   # delayed 30+ min
    (df_delta["CANCELLED"] == 1)             # or cancelled
].copy()

# Classify each disruption using our function from Notebook 1
def classify_disruption(row):
    if row["CANCELLED"] == 1:
        return "CANCELLATION", 5
    elif row["DEPARTURE_DELAY"] >= 120:
        return "DELAY_CRITICAL", 3
    elif row["DEPARTURE_DELAY"] >= 60:
        return "DELAY_MAJOR", 2
    elif row["DEPARTURE_DELAY"] >= 30:
        return "DELAY_MINOR", 1
    else:
        return "ON_TIME", 0

# Apply classification to every row
df_disrupted[["disruption_type", "severity"]] = df_disrupted.apply(
    classify_disruption, axis=1, result_type="expand"
)

print(f"Total Delta flights       : {len(df_delta):,}")
print(f"Disrupted flights         : {len(df_disrupted):,}")
print(f"Disruption rate           : {len(df_disrupted)/len(df_delta)*100:.1f}%")
print(f"\n=== Disruption Breakdown ===")
print(df_disrupted["disruption_type"].value_counts())
print(f"\n=== By Severity ===")
print(df_disrupted["severity"].value_counts().sort_index(ascending=False))

Total Delta flights       : 875,881
Disrupted flights         : 75,287
Disruption rate           : 8.6%

=== Disruption Breakdown ===
disruption_type
DELAY_MINOR       35651
DELAY_MAJOR       21655
DELAY_CRITICAL    14157
CANCELLATION       3824
Name: count, dtype: int64

=== By Severity ===
severity
5     3824
3    14157
2    21655
1    35651
Name: count, dtype: int64


In [13]:
Business insight (this is what matters)

While minor delays represent the largest volume, 
high-severity disruptions (major delays, 
critical delays, and cancellations) account for a substantial 
portion of events and should be prioritized due to their higher 
operational and customer impact 


axis=1 applies the function to each row. 
But you did not explain why axis=0 fails here. 
The reason: axis=0 would pass each entire 
column as a series to your function. 
Your function expects row["CANCELLED"] 
and row["DEPARTURE_DELAY"] — individual row values.
A column cannot provide that. 
Always think about what your function receives as input.#

SyntaxError: invalid syntax (1121690849.py, line 1)

In [17]:
# Cell 5 — Build mock passenger database (improved)
import numpy as np

np.random.seed(42)

df_sample = df_disrupted.sample(n=1000, random_state=42).copy()
df_sample = df_sample.reset_index(drop=True)

loyalty_tiers = ["GENERAL", "SILVER", "GOLD", "PLATINUM", "DIAMOND"]
tier_weights  = [0.50, 0.25, 0.15, 0.07, 0.03]

passengers = []
for idx, flight in df_sample.iterrows():
    n_passengers = np.random.randint(1, 4)
    for p in range(n_passengers):
        passengers.append({
            "passenger_id":   f"PAX-{idx:04d}-{p}",
            "flight_id":      f"{flight['AIRLINE']}{int(flight['FLIGHT_NUMBER'])}",
            "origin":         flight["ORIGIN_AIRPORT"],
            "destination":    flight["DESTINATION_AIRPORT"],
            "disruption_type":flight["disruption_type"],
            "severity":       flight["severity"],
            "loyalty_tier":   np.random.choice(loyalty_tiers, p=tier_weights),
            "special_need":   np.random.choice(
                                  ["NONE", "MEDICAL", "WHEELCHAIR", "INFANT"],
                                  p=[0.60, 0.15, 0.15, 0.10]
                              ),
            "has_connection": np.random.choice([True, False], p=[0.35, 0.65]),
        })

df_passengers = pd.DataFrame(passengers)

print(f"Passengers generated      : {len(df_passengers):,}")
print(f"\n=== Loyalty Tier Distribution ===")
print(df_passengers["loyalty_tier"].value_counts())
print(f"\n=== Special Needs ===")
print(df_passengers["special_need"].value_counts())
print(f"\n=== Has Connecting Flight ===")
print(df_passengers["has_connection"].value_counts())

Passengers generated      : 1,974

=== Loyalty Tier Distribution ===
loyalty_tier
GENERAL     964
SILVER      532
GOLD        297
PLATINUM    132
DIAMOND      49
Name: count, dtype: int64

=== Special Needs ===
special_need
NONE          1202
WHEELCHAIR     294
MEDICAL        294
INFANT         184
Name: count, dtype: int64

=== Has Connecting Flight ===
has_connection
False    1281
True      693
Name: count, dtype: int64


In [25]:
from pathlib import Path

BASE_DIR = Path("../").resolve()
DB_DIR   = BASE_DIR / "database"

print(f"Project root : {BASE_DIR}")
print(f"Database dir : {DB_DIR}")
print(f"Folder exists: {DB_DIR.exists()}")

# Create it if missing
DB_DIR.mkdir(parents=True, exist_ok=True)
print(f"Folder ready : {DB_DIR.exists()}")

Project root : C:\Users\HP\Desktop\delta_disruption_project
Database dir : C:\Users\HP\Desktop\delta_disruption_project\database
Folder exists: False
Folder ready : True


In [27]:
# Cell 6 — Save to SQLite database
import sqlite3
import pandas as pd
from pathlib import Path

# Build path and convert backslashes to forward slashes
BASE_DIR = Path("../").resolve()
DB_PATH  = BASE_DIR / "database" / "delta_disruption.db"
DB_STR   = str(DB_PATH).replace("\\", "/")

print(f"Saving to: {DB_STR}")

# Use sqlite3 directly — no SQLAlchemy path issues
conn = sqlite3.connect(DB_STR)

# Save both dataframes
df_disrupted.to_sql("disrupted_flights", conn,
                    if_exists="replace", index=False)

df_passengers.to_sql("passengers", conn,
                     if_exists="replace", index=False)

conn.commit()
conn.close()

print("✓ Tables saved to delta_disruption.db")
print(f"  disrupted_flights : {len(df_disrupted):,} rows")
print(f"  passengers        : {len(df_passengers):,} rows")

# Verify by reading back
conn = sqlite3.connect(DB_STR)
count = pd.read_sql("SELECT COUNT(*) as total FROM passengers", conn)
print(f"\n✓ Verified — passengers in DB: {count['total'][0]:,}")
conn.close()

Saving to: C:/Users/HP/Desktop/delta_disruption_project/database/delta_disruption.db
✓ Tables saved to delta_disruption.db
  disrupted_flights : 75,287 rows
  passengers        : 1,974 rows

✓ Verified — passengers in DB: 1,974
